# MVP Demo Runbook — 2026-05-29 (with 2026-06-30 comparison)

This notebook runs the demo case **step by step**. It does not replace the PPT; it produces the numbers the PPT walks through.

Order:

1. **Market regime** — UMD / Daniel–Moskowitz context (deterministic)
2. **Quant signals** — scorecard values, thresholds, trigger states (deterministic)
3. **Structural & mechanical unwind** — mechanism scenarios, concentration, footprint (deterministic)
4. **AI evidence layer** — interpretation of the deterministic layers + evidence challenge
5. **Final PM read** — the combined decision-support read

Default run is **live DeepSeek** (`USE_LLM=True`). Missing key / HTTP failure / schema validation still fail closed to deterministic text. Set `USE_LLM=False` in the setup cell for the fully offline deterministic run.

## Runbook map

| Step | Layer | What you see |
| --- | --- | --- |
| 1 | Market regime | UMD/DM state, drawdown/recovery/volatility conditions, state-conditioned history |
| 2 | Quant signals | 4 deterministic scorecard indicators: value, threshold, status, change vs comparison |
| 3 | Structural / mechanical | 3 mechanism scenarios, theme concentration, factor footprint, turnover, absorption |
| 4 | AI evidence layer | Narrative interpretation + supporting / unconfirmed / premature evidence |
| 5 | Final PM read | One integrated read: current state, vulnerability, why not act yet, next checks |
| 6 | 5/29 vs 6/30 | Same pipeline, two dates: mechanism, fragility, evidence, LLM read |
| 7 | GDELT news read (optional) | Live DeepSeek narrative over GDELT headlines; gated by quant + mechanism triggers |
| 8 | Investigation agent | Observe → Decide → Act → Evaluate → Decide again → Stop; cannot change the risk state |

Technical modes, versions, and cache boundaries are in the appendix at the bottom.


In [1]:
%run -i demo_setup.py


In [2]:
# Run the full MVP once; every step below only renders its layer.
result = run_with_retry(CONFIG)

evidence = result.deterministic_input
unwind = result.unwind
mechanical = result.mechanical_unwind
interpretation = result.interpretation
pm = result.pm_response

positioning = build_positioning_snapshot(
    as_of_date=CONFIG.as_of_date,
    context_elevated=bool(evidence.triggered_quant_signals),
    processed_dir=CONFIG.processed_dir,
)
positioning_proxies = public_positioning_proxy_items(positioning)

print("MVP demo run complete —", CONFIG.as_of_date, "| fingerprint:", result.full_run_fingerprint)

MVP demo run complete — 2026-05-29 | fingerprint: cae6c2819e643ed7


## Step 1 — Market regime (deterministic)

UMD / Daniel–Moskowitz state and market conditions are **comparison context only**. They are never merged into a PM-book crash probability.

In [3]:
primary = build_primary_assessment(
    as_of_date=pd.Timestamp(CONFIG.as_of_date),
    horizon=CONFIG.horizon_days,
    processed_dir=CONFIG.processed_dir,
)
factors = pd.read_parquet(CONFIG.processed_dir / "french_research_factors_daily.parquet")
regime = build_regime_history(factors)
row = regime.loc[regime["date"].eq(pd.Timestamp(CONFIG.as_of_date))].iloc[0]

display(*render_market_regime(primary, row, evidence, CONFIG.as_of_date))

### Market regime — 2026-05-29

| Field | Value | Note |
| --- | --- | --- |
| UMD / DM state | `normal` | comparison context only |
| Bear state (504d market return < 0) | `False` |  |
| Market return, 504d | `0.4863` |  |
| Market drawdown | `0.0000` |  |
| Recent min drawdown, 126d | `-0.0946` |  |
| Recovery from trough, 126d | `0.1948` |  |
| High volatility, 21d | `False` |  |
| High-vol recovery state | `False` |  |
| Rate regime | `tightening` |  |


**UMD comparison context** (descriptive history, not PM-book probability):

| State | Sample | Tail-loss freq | Mean fwd return | 5th pct |
| --- | --- | --- | --- | --- |
| `all` | 25629 | 0.0505 | 0.0054 | -0.0587 |
| `normal` | 21132 | 0.0336 | 0.0075 | -0.0464 |
| `bear_low_volatility` | 3116 | 0.0815 | -0.0012 | -0.0791 |
| `panic_elevated` | 1381 | 0.2390 | -0.0130 | -0.1817 |


## Step 2 — Quant signals (deterministic scorecard)

Four indicators with explicit values and thresholds. Status is **triggered / not triggered**, not a probability.

In [4]:
signals = list(evidence.triggered_quant_signals) + list(evidence.non_triggered_relevant_signals)
display(render_quant_signals(signals, evidence.as_of_date))

### Quant signals — 2026-05-29

| Metric | Value | Threshold | Status | Δ vs comparison | Read |
| --- | --- | --- | --- | --- | --- |
| `high_volatility_recovery` | `0.0000` | `1.0000` | `not_triggered` | `+0.0000` | One composite macro gate replaces separate drawdown, recovery, and volatility alerts. It requires Phase 1 early recovery and high realized volatility to be true together. |
| `short_minus_long_beta_gap` | `-2.0677` | `0.2490` | `not_triggered` | `-0.1726` | Positive and unusually high short-underlying minus long beta indicates that a market rebound can squeeze the recent-loser leg. Threshold: prior-only 80th percentile from 2281 observations; raw historical threshold=0.248979. |
| `portfolio_drawdown` | `-0.0752` | `-0.1741` | `not_triggered` | `-0.0752` | Long-short wealth relative to its highest level in the prior 63 trading days is compared with its own prior-only left-tail history. The threshold can never be looser than -20%, and drawdowns shallower than 5% are not material. Threshold: prior-only 20th percentile from 2281 observations; raw historical threshold=-0.174081. |
| `short_loss_in_recovery` | `0.1551` | `0.2504` | `not_triggered` | `+0.0046` | Trailing 21-day short loss magnitude is the sum of negative signed short contributions. It triggers only when Phase 1 early recovery is active and the loss reaches the threshold. Threshold: prior-only 80th percentile from 2323 observations; raw historical threshold=0.25042. |

## Step 3 — Structural & mechanical unwind (deterministic)

Three independent mechanism lenses, then concentration and market-footprint proxies. These layers are read separately from the quant scorecard.

In [5]:
display(*render_structural_mechanical(unwind, mechanical))

### Mechanism scenarios — 2026-05-29

| Mechanism scenario | Status | Read |
| --- | --- | --- |
| `bear_market_recovery_crash` | `watch` | Some bear-market-recovery preconditions are present, but the three-part mechanism is not confirmed. |
| `short_book_reversal_crash` | `not_confirmed` | Short-book reversal conditions are not present. |
| `crowded_theme_unwind` | `triggered` | A pre-event correlated long cluster is concentrated and is experiencing broad, extreme, loss- or volume-confirmed selling. |

### Theme concentration

| Field | Value |
| --- | --- |
| Cluster | `CIEN, COHR, LITE` |
| Active long symbols | `CIEN, COHR, ECHO, FIX, LITE, MU, SNDK, STX, TER, WDC` |
| Cluster exposure share | `0.3000` |
| Avg residual correlation | `0.7256` |
| 5d residual loss | `0.0738` |
| 5d abnormal volume share | `0.6667` |
| Concentration trigger | `True` |


### Mechanical footprint

| Field | Value |
| --- | --- |
| Unwind state | `FRAGILITY_BUILDING` |
| Control spec | `mom_vol` |
| Factor footprint R² | `0.1251` |
| Factor footprint percentile | `0.7689` |
| Extreme turnover ratio | `1.1125` |
| Extreme turnover percentile | `0.8566` |
| Liquidity absorption failure | `False` |
| Absorption percentile | `0.2709` |


### Unwind scorecard (6 rows)

| Metric | Value | Threshold | Triggered | Severity | Explanation |
| --- | --- | --- | --- | --- | --- |
| `portfolio_concentration` | `19.5188` | `19.7888` | `True` | `high` | Gross-normalized effective bets use drifted beginning-of-day exposure; lower values indicate greater concentration. |
| `momentum_breadth_deterioration` | `0.6580` | `0.5558` | `False` | `normal` | The share of the eligible universe with positive 12-1 momentum is compared with its strictly prior monthly history. |
| `synchronous_winner_liquidation` | `-0.0059` | `0.0192` | `False` | `normal` | An extreme five-day lagged-beta-adjusted long loss must coincide with broad active-long declines. |
| `cross_sectional_reversal` | `-0.0347` | `0.0331` | `False` | `normal` | Positive short-underlying-minus-long return means prior losers outperformed prior winners over five trading days. |
| `liquidity_amplification_proxy` | `0.3000` | `0.5000` | `False` | `normal` | This public-data proxy counts active long names falling while five-day volume exceeds each name's prior-only threshold. |
| `fundamental_anchor` | `unavailable` | `coverage-gated sign-vote rule` | `None` | `unavailable` | Revenue acceleration, applicable operating-margin change, and optional EPS acceleration provide a lightweight sign-based anchor. |

## Step 4 — AI evidence layer

The AI layer interprets the deterministic layers and organizes supporting / contradicting / missing evidence. It **cannot change** any deterministic value, threshold, or trigger.

In [6]:
display(render_ai_evidence(evidence, interpretation, CASE_PACKS, CONFIG))

### AI evidence layer — live DeepSeek (condensed)

**Read:** Crowded-theme unwind triggered; short-side crowding plausible; no broad mechanical unwind confirmed; fundamentals mixed.

**Key counter-evidence (8):**

- `csu-2026-05-29-001` — NVIDIA Announces Financial Results for First Quarter Fiscal 2027 — NVIDIA Investor Relations
- `csu-2026-05-29-006` — Cisco Reports Third Quarter Earnings — Cisco Investor Relations
- `csu-2026-05-29-007` — Arista Networks Reports First Quarter 2026 Financial Results — Arista Networks Investor Relations
- `csu-2026-05-29-008` — Coherent Reports Third Quarter Fiscal 2026 Results — Coherent Investor Relations
- `csu-2026-05-29-009` — Lumentum Announces Third Quarter Fiscal 2026 Financial Results — Lumentum Investor Relations
- `csu-2026-05-29-010` — Applied Materials Announces Second Quarter 2026 Results — Applied Materials Investor Relations
- `csu-2026-05-29-011` — Lam Research Reports Financial Results for Quarter Ended March 29, 2026 — Lam Research Investor Relations
- `csu-2026-05-29-012` — TSMC April 2026 Revenue Report — TSMC

**Key missing:**

- No direct evidence on investor identity or leverage in the short basket; public proxies are context only.
- No direct evidence of forced deleveraging or margin pressure; forced deleveraging remains unconfirmed.

<details><summary>Supporting · Monitoring · Invalidation · Quality</summary>

**PM interpretation:** “Crowded-theme unwind is triggered, indicating potential momentum tail risk. Short-side crowding is plausible given elevated public short-interest proxies, but this does not establish active covering or forced deleveraging. No broad mechanical unwind is confirmed; factor footprint is not elevated. Fundamental earnings reports from major tech names are contradicting the unwind narrative, suggesting a fundamental repricing may be at play. Monitoring should focus on whether short-loss-in-recovery escalates while high-volatility-recovery remains untriggered, and whether factor propagation emerges. Forced deleveraging remains unconfirmed.”

**Supporting:**

- `csu-2026-05-29-013` — Tech stocks see largest hedge fund selloff in decade: Goldman Sachs — Investing.com reporting Goldman Sachs Prime Book data

**Monitoring questions:**

- Is the short-loss-in-recovery signal approaching its threshold while the high-volatility-recovery signal remains untriggered?
- Is the factor-footprint status showing signs of elevation while aligned turnover remains elevated?
- Is the liquidity-absorption-failure flag at risk of turning true?

**Invalidation conditions:**

- If the short-loss-in-recovery signal triggers while the high-volatility-recovery signal remains untriggered, the recovery-crash scenario becomes more plausible.
- If the factor-footprint status turns elevated while the aligned-turnover status remains elevated, a broad mechanical unwind is more likely.
- If the liquidity-absorption-failure flag becomes true, the unwind risk escalates.

**Evidence quality:** `available` · `evidence-interpretation-prompt-v8`

</details>

<details><summary>Deterministic adapter warnings (technical)</summary>

- The cached corpus is small and may omit relevant contradictory evidence.
- Generic macro context does not establish momentum-specific causality.
- Evidence cannot change deterministic metrics, thresholds, triggered states, or create a risk score.
- No composite deterministic score is defined; the adapter preserves the four indicator states and leaves deterministic_score null.
- The legacy Phase 6 adapter contract contains Phase 5A feasibility metadata only and does not embed the separate Phase 5 unwind scorecard; the notebook renders that deterministic assessment alongside this card. Its fundamental row remains unavailable unless exact-date company coverage is supplied.

</details>

<details><summary><strong>Frozen evidence challenge (2026-05-29 pack)</strong> — supporting / unconfirmed / premature</summary>

## What is supported

- Structured crowded-theme unwind and concentration stress in the PM book.
- Elevated turnover and concentrated pressure, with no sign that market liquidity is failing.
- Market-recovery component in retrieved text (`CSU-2026-015`).
- Contradicting operating strength at a cluster name (`CSU-2026-008`).

## What remains unconfirmed

- Forced deleveraging, financing pressure, or dealer-inventory stress.
- Factor propagation beyond the detected cluster.
- Liquidity-absorption failure.
- Complete DM sequence (panic + loser-leg rebound + short-leg loss).
- Completed fundamental valuation or earnings reprice.
- Stance-confirmed citation of contextual items `CSU-2026-013`, `CSU-2026-004`, `CSU-2026-005` (MVP citation limitation).

## Why broad action may still be premature

A crowded-theme signal warrants focused review, but broad automatic de-risking is still premature: selling is being absorbed, stress has not spread beyond the cluster, the recovery-crash and fundamental lenses remain weak, and positioning evidence is not strong enough to confirm the narrative. Any assessment of the later semiconductor selloff requires a refreshed portfolio snapshot and evidence from the same cutoff.

</details>

## Step 5 — Final PM read

All layers collapse into one decision-support read: what is happening, where the risk sits, why broad action is premature, and what would change the reading.

In [7]:
display(render_final_pm_read(evidence, unwind, mechanical, pm, CONFIG.as_of_date))

### Final PM read — 2026-05-29

| Layer | Read |
| --- | --- |
| UMD / market context | `normal` (comparison only) |
| Scorecard triggers | 0 |
| Recovery crash | `watch` |
| Short-book reversal | `not_confirmed` |
| Crowded theme unwind | `triggered` |
| Momentum tail-risk state | `potential momentum tail risk` |
| Classification | `crowded_momentum_unwind` |

**Current read:** “No deterministic escalation signals are active; the crowded theme unwind mechanism is triggered, but we are not seeing a broad mechanical unwind or a confirmed short-book reversal crash. Maintain posture and monitor the short basket.”

**Main vulnerability:** “The main vulnerability is rebound-sensitive shorts in the momentum-loser basket, which could be crowded given the elevated short-interest proxy in the public data universe; this path is not active today but is worth watching.”

**Why not act yet:** “We are not acting yet because the deterministic signals are not triggered and the short-book reversal crash is not confirmed. The elevated short-interest proxy is contextual support for crowding but not proof of covering or forced deleveraging. We prefer to maintain posture and monitor rather than preemptively de-risk.”

**What would change the reading:**

- If the short-book reversal crash mechanism moves from watch to triggered, or if short losses in a recovery regime trigger, we would escalate to a review of rebound-sensitive shorts and a loser-rally stress scenario.
- If the short-minus-long beta gap widens into a triggered state or portfolio drawdown breaches its threshold, we would consider pausing incremental risk and reviewing unintended beta.

**Conditional response:**

- If the crowded theme unwind continues, we would first review short concentration and pause incremental risk pending PM review.
- If a recovery move confirms, we would then review rebound-sensitive shorts and run a loser-rally stress scenario.

<details><summary>Response categories (bounded menu)</summary>

- Maintain the current posture and monitor the short basket
- Check whether short concentration is amplifying the move
- Review rebound-sensitive shorts if a recovery move confirms

</details>

*Mode: live DeepSeek (`pm-response-prompt-v5`)*

## Step 6 — 2026-05-29 vs 2026-06-30 (same pipeline)

Same rules, two dates, both with exact-date evidence replay and live DeepSeek interpretation. The point is not a forecast: it shows how the mechanism read and the fragility footprint changed between the two cutoffs.

In [8]:
# Run 2026-06-30 through the identical pipeline (live DeepSeek).
config_630 = MVPConfig(
    as_of_date="2026-06-30",
    compare_to_date="2026-05-29",
    threshold_profile="default",
    horizon_days=20,
    use_llm=USE_LLM,
)
result_630 = run_with_retry(config_630)

ev6 = result_630.deterministic_input
un6 = result_630.unwind
mech6 = result_630.mechanical_unwind
interp6 = result_630.interpretation

display(render_comparison(evidence, unwind, mechanical, interpretation, ev6, un6, mech6, interp6))

### What changed from 2026-05-29 to 2026-06-30

| Dimension | 2026-05-29 | 2026-06-30 |
| --- | --- | --- |
| Overall state | normal | normal |
| Quant triggers | 0 | 0 |
| Scenario classification | crowded_momentum_unwind | normal_drawdown |
| Active mechanisms | crowded_theme_unwind | none |
| Theme cluster | CIEN, COHR, LITE | MU, STX, WDC |
| Theme trigger | True | False |
| Mechanical state | FRAGILITY_BUILDING | FRAGILITY_BUILDING |
| Factor footprint pct | 0.7689 | 0.8606 |
| Extreme turnover pct | 0.8566 | 0.0040 |
| Liquidity absorption failure | False | True |
| Evidence items | 15 | 3 |
| Evidence mode | live DeepSeek | live DeepSeek |
| LLM narrative | Crowded-theme unwind triggered; short-side crowding plausible; no broad mechanical unwind confirmed; fundamentals mixed. | Potential momentum tail risk; no confirmed escalation; mechanical fragility building with elevated factor footprint and short-side crowding plausible. |
| PM posture | escalate_for_pm_review | focused_review_and_monitor |

## Step 7 — GDELT news read (optional)

An optional live-DeepSeek layer that explains the already-gated momentum-risk state using contemporaneous GDELT headlines. It is gated by quantitative signals **and** structural mechanism statuses, and it fails closed if the cached titles or the API are unavailable.

In [9]:
# Optional GDELT news-read layer (live DeepSeek; fails closed offline).
from src.evidence.gdelt_evidence import (
    load_gdelt_titles,
    retrieve_gdelt_evidence,
    active_triggers_from_state,
    no_active_trigger_message,
    no_evidence_message,
)
from src.evidence.deepseek_explainer import explain_risk_with_deepseek

gdelt_titles = load_gdelt_titles()


def gdelt_read(run_result, label):
    ev = run_result.deterministic_input
    sigs = list(ev.triggered_quant_signals) + list(ev.non_triggered_relevant_signals)
    triggers = active_triggers_from_state(sigs, run_result.unwind.mechanism_scenarios)
    if not triggers:
        return None, no_active_trigger_message()
    evidence = retrieve_gdelt_evidence(
        gdelt_titles,
        as_of_date=ev.as_of_date,
        active_triggers=triggers,
    )
    if evidence.empty:
        return None, no_evidence_message()
    explain = explain_risk_with_deepseek(
        triggers,
        evidence,
        as_of_date=ev.as_of_date,
        cache_dir=ROOT / "outputs" / "gdelt_evidence_cache",
    )
    return (triggers, evidence, explain), None


reads = {}
for label, rr in [("2026-05-29", result), ("2026-06-30", result_630)]:
    try:
        payload, message = gdelt_read(rr, label)
        reads[label] = (payload, message)
    except Exception as exc:
        reads[label] = (None, f"{type(exc).__name__}: {exc}")

for label, (payload, message) in reads.items():
    display(render_gdelt_read(label, payload, message))

**GDELT — 2026-05-29** (condensed)

**Triggers:** bear_market_recovery_crash (partial), crowded_theme_unwind (triggered)

**Read:** The momentum tail-risk monitor flags a partially triggered 'bear_market_recovery_crash' and a fully triggered 'crowded_theme_unwind' as of 2026-05-29. The former is in a monitoring state, not confirmed, while the latter is active.

**PM takeaway:** The monitor suggests elevated risk of a crowded trade unwind, which could exacerbate any market weakness. The partial 'bear_market_recovery_crash' indicates that while recession fears are present, they have not yet materialized into a confirmed crash. Stay alert for signs of positioning stress, but avoid overreacting to headlines alone.

**Top evidence:**

- `E1` (2026-05-20) **How to recession - proof your life** — cnn.com · matched: `bear_market_recovery_crash`
- `E2` (2026-05-13) **Citadel Quant Chief : AI New Market Paradox Faster Information , More Crowded Trades :** — hedgeco.net · matched: `crowded_theme_unwind`
- `E3` (2026-05-10) **3 Steps to Get Your Retirement Portfolio Recession - Ready** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E4` (2026-05-10) **Stock Market Crash : The Best Dividend Stocks to Buy Right Now** — finance.yahoo.com · matched: `bear_market_recovery_crash`

<details><summary>Recent narrative · Mechanism · Limitations · All evidence</summary>

**Recent narrative:** Recent GDELT records show a mix of recession-related advice and market commentary. Articles like 'How to recession-proof your life' [E1] and '3 Steps to Get Your Retirement Portfolio Recession-Ready' [E3] suggest elevated public concern about a potential downturn. Mentions of 'market crash' in a stock-picking context [E4] and specific selloffs in individual stocks [E5][E6] indicate pockets of weakness. Concurrently, a piece on crowded trades [E2] highlights that AI-driven information speed is leading to more crowded positioning, which is a classic setup for sharp unwinds.

**Momentum mechanism:** The 'bear_market_recovery_crash' trigger is partially set, likely because while there is chatter about recession and selloffs, there is no clear evidence of a broad market crash or a failed recovery. The 'crowded_theme_unwind' trigger is fully triggered, as the article [E2] directly discusses how faster information leads to more crowded trades, increasing the risk of sudden, sharp reversals when positioning becomes too one-sided. This aligns with the momentum mechanism where crowded trades can unwind violently, amplifying downward moves.

**Limitations:** The evidence is limited to a few news articles and may not capture the full market context. The 'bear_market_recovery_crash' trigger is only partial, and the articles do not confirm a crash or a failed recovery. The 'crowded_theme_unwind' trigger is based on a single opinion piece, which is weak evidence for a systemic unwind. No causality is implied.

**All evidence:**

- `E1` (2026-05-20) **How to recession - proof your life** — cnn.com · matched: `bear_market_recovery_crash`
- `E2` (2026-05-13) **Citadel Quant Chief : AI New Market Paradox Faster Information , More Crowded Trades :** — hedgeco.net · matched: `crowded_theme_unwind`
- `E3` (2026-05-10) **3 Steps to Get Your Retirement Portfolio Recession - Ready** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E4` (2026-05-10) **Stock Market Crash : The Best Dividend Stocks to Buy Right Now** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E5` (2026-05-13) **Is the Selloff in OrthoPediatrics Corp . ( KIDS ) Creating a Buying Opportunity ?** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E6` (2026-05-13) **Here Why Outset Medical ( OM ) Rebound Sharply After Last Quarter Selloff** — insidermonkey.com · matched: `bear_market_recovery_crash`

</details>

**GDELT — 2026-06-30** (condensed)

**Triggers:** short_loss_in_recovery (partial), bear_market_recovery_crash (partial), short_book_reversal_crash (partial), crowded_theme_unwind (partial)

**Read:** The momentum tail-risk monitor shows partial triggers for 'short_loss_in_recovery' (progress 73.6%) and 'bear_market_recovery_crash', 'short_book_reversal_crash', and 'crowded_theme_unwind' (all partial). This indicates a fragile market state with elevated risk of a sharp reversal or crash, though not yet confirmed.

**PM takeaway:** The market is in a fragile state with elevated crash risk. Monitor for further deterioration in tech and crowded trades. The partial triggers warrant caution but not immediate action.

**Top evidence:**

- `E1` (2026-06-30) **Could the SpaceX , Anthropic , and OpenAI IPOs Trigger a 40 % Stock Market Crash ? Here What the Data Says .** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E2` (2026-06-26) **Why Investors Keep Paying Too Much For Crowded Trades** — forbes.com · matched: `crowded_theme_unwind`
- `E3` (2026-06-24) **A Global Rout in Tech Stocks Is at the Epicenter of a U . S . Selloff . What to Watch Next .** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E4` (2026-06-24) **AI Stock Selloff Hits Nasdaq 2 . 2 %: KOSPI Circuit Breaker Signals a Divided Market** — techtimes.com · matched: `bear_market_recovery_crash`

<details><summary>Recent narrative · Mechanism · Limitations · All evidence</summary>

**Recent narrative:** In late June 2026, public discourse is dominated by crash warnings and selloff narratives. Articles from Yahoo Finance and Fool.com speculate on a potential 40% crash triggered by major IPOs [E1] and debate the likelihood of a 2026 crash [E5][E6]. A global tech rout is described as the epicenter of a U.S. selloff [E3], with AI stocks leading a Nasdaq drop of 2.2% and triggering a KOSPI circuit breaker [E4]. A semiconductor selloff is attributed to Broadcom's disappointing AI outlook [E11]. Concurrently, there is commentary on crowded trades, noting investors overpaying for popular positions [E2], and a risk-off tone in weekly trending stocks [E10]. Even a piece on financial stocks to buy in a crash [E7] and a discussion of a bear market [E12] reflect heightened anxiety. Earlier in June, a market crash on good jobs news was analyzed [E8], and advice on preparing for a recession [E9] adds to the cautious sentiment.

**Momentum mechanism:** The partial triggers suggest that negative momentum is building. The 'short_loss_in_recovery' trigger, with 73.6% progress, indicates that short positions may be under pressure as the market attempts to recover, but the pervasive crash talk and selloffs could force a reversal. The 'bear_market_recovery_crash' trigger is supported by multiple articles discussing crash risks and actual selloffs, implying that any recovery attempt may be fragile. The 'crowded_theme_unwind' trigger is backed by commentary on crowded trades, suggesting that popular positions (e.g., AI stocks) are vulnerable to sharp unwinding. These mechanisms are interconnected: a selloff in crowded tech trades could trigger a broader market decline, exacerbating short losses and leading to a crash scenario.

**Limitations:** The evidence is largely from financial media and opinion pieces, not official market data. The articles are speculative and may not reflect actual market conditions. The trigger status is partial, so this is not a confirmed alert. No causal link is established between the news and market movements.

**All evidence:**

- `E1` (2026-06-30) **Could the SpaceX , Anthropic , and OpenAI IPOs Trigger a 40 % Stock Market Crash ? Here What the Data Says .** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E2` (2026-06-26) **Why Investors Keep Paying Too Much For Crowded Trades** — forbes.com · matched: `crowded_theme_unwind`
- `E3` (2026-06-24) **A Global Rout in Tech Stocks Is at the Epicenter of a U . S . Selloff . What to Watch Next .** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E4` (2026-06-24) **AI Stock Selloff Hits Nasdaq 2 . 2 %: KOSPI Circuit Breaker Signals a Divided Market** — techtimes.com · matched: `bear_market_recovery_crash`
- `E5` (2026-06-22) **Will the Stock Market Crash in 2026 ? History Shows This Is the Smartest Way to Prepare .** — fool.com · matched: `bear_market_recovery_crash`
- `E6` (2026-06-18) **Is a Stock Market Crash Coming in 2026 ? History Has Good and Bad News for Investors .** — fool.com · matched: `bear_market_recovery_crash`
- `E7` (2026-06-21) **Market Crash : The Financial Stocks Id Buy Without Losing Sleep** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E8` (2026-06-10) **Why did the stock market crash on good jobs news ? Glenn Beck unpacks the sick game Wall Street is playing** — theblaze.com · matched: `bear_market_recovery_crash`
- `E9` (2026-06-01) **5 moves retirees should make before a recession hits so youre never forced to sell investments at a loss** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E10` (2026-06-07) **Trending stocks this week amid risk - off selloff on Wall Street ( COMP : IND :) ( DJI :)** — seekingalpha.com · matched: `bear_market_recovery_crash`
- `E11` (2026-06-04) **Micron Drops 7 % as Broadcom Disappointing AI Outlook Triggers a Semiconductor Selloff** — finance.yahoo.com · matched: `bear_market_recovery_crash`
- `E12` (2026-06-03) **There Are Now Over 10 Stocks in the S & P 500 Index With Market Caps Exceeding $1 Trillion . This Is the Best One to Own in a Bear Market . It Not Remotely Close and the Stock Is on Sale .** — finance.yahoo.com · matched: `bear_market_recovery_crash`

</details>

## Step 8 — Investigation agent loop

The deterministic engine already owns the risk state. This cell runs a thin hand-written loop:

```text
Observe → Decide → Act → Evaluate → Decide again → Stop
```

It searches one mechanism at a time, allows at most one narrower follow-up while that hypothesis is unresolved, then moves to the next active mechanism. It cannot change triggers, thresholds, scores, or issue a trade.


In [ ]:
agent_result = run_investigation_demo(result)
display(Markdown(f"```text\n{agent_result.report}\n```"))


## Cross-case comparison (one screen)

Same rules, three different conclusions. Values come from repository outputs, not hard-coded demo claims.

In [10]:
display(Markdown((CASE_PACKS["cross_case"]).read_text()))

# Cross-case comparison

Derived from repository case packs and `run_mvp` outputs. Mechanism labels are descriptive reads, not crash probabilities.

| Question | Current semi case (2026-05-29) | 2020 validation (2020-03-24) | 2024 quiet control (2024-01-05) |
| --- | --- | --- | --- |
| Recovery mechanism (Daniel–Moskowitz) | Partial / watch — recovery text without panic, loser rebound, or short-loss confirmation | Strongly present — `bear_market_recovery_crash` triggered; panic, severe drawdown, high vol, recovery aligned | Not present as a completed setup — recovery precondition only; severe drawdown and high vol unmet |
| Crowded unwind evidence (Khandani–Lo) | Contextual / partially supported — `crowded_theme_unwind` triggered; concentration and potential momentum tail risk, while selling is still being absorbed | Secondary / unconfirmed — liquidity facilities ≠ crowded positioning | Limited — crowded-theme scenario not confirmed; trading footprint is normal |
| Short-leg pressure | Contained on scorecard; risk sits more in long-side crowding | Severe — `short_loss_in_recovery` and beta-gap triggered; short-book reversal on watch | Contained — short-loss, beta-gap, and short-book reversal not triggered |
| Evidence confidence | Mixed — localized crowding supported; broad crash and forced unwind unconfirmed | Historically coherent — mechanism indicators line up with a known reversal episode | Low-risk / quiet — ordinary macro context; no confirmed crash channel |
| PM workflow | Monitor and investigate concentrated / theme exposures | Escalate review of recovery-crash and short-basket channels | Maintain monitoring — escalation not justified |

## How to read the table

1. **Current semi** is the primary live-style product demo: localized crowding pressure without a completed recovery crash.
2. **2020** shows that when a historically important momentum reversal occurred, the recovery-crash indicators behaved coherently.
3. **2024** shows the same rules staying quiet when the mechanism is incomplete.

Sources:

- `outputs/current_semi_unwind/pm_case_read.md`
- `outputs/current_semi_unwind/mechanism_comparison.md`
- `outputs/march_2020_reference/pm_case_read.md`
- `outputs/march_2020_reference/mechanism_comparison.md`
- `outputs/quiet_control_2024/pm_case_read.md`
- `outputs/quiet_control_2024/mechanism_comparison.md`
- `outputs/research_validation/episode_fingerprints.md`


<details>
<summary><strong>Technical appendix — not needed during the demo</strong></summary>

- **Run modes:** This runbook defaults to `USE_LLM=True` (live DeepSeek) for the PPT demo; it requires `DEEPSEEK_API_KEY` in `.env`. Set `USE_LLM=False` for offline deterministic Evidence Card + PM narrative, no API call. Missing key / HTTP failure / schema validation fails closed to deterministic text and never rewrites metrics.
- **Evidence layer:** exact-date validated cache replay via `src/evidence/research_preview.py`; missing, malformed, future-dated, or post-cutoff evidence fails closed to `unavailable`.
- **Evidence cache coverage:**
- **GDELT news read (optional):** live DeepSeek explanation over cached GDELT titles (`data/raw/gdelt_phase2/`); gated by quant + mechanism triggers; fails closed if data/API are unavailable. 2026-05-29 now has an exact-date replay of 15 human-reviewed CSU records (hash-verified, cutoff-valid); 2026-06-30 has the bundled 3-item minimal cache. Missing or invalid caches fail closed to `unavailable`, and evidence can never change deterministic fields.
- **Versions:** deterministic evidence interpretation `deterministic-evidence-interpretation-v2`; deterministic PM response `deterministic-pm-response-v1`; live prompts `evidence-interpretation-prompt-v8` / `pm-response-prompt-v5`.
- **Quant components:** `src/mvp/evidence_card.py`, `src/mvp/pipeline.py`, `src/monitoring/scorecard.py`, `src/monitoring/unwind_monitor.py`, `src/regime/market_state.py`.
- **Investigation agent:** `src/agent.py` (`run_investigation_loop`) observes the frozen risk state, chooses a mechanism search, executes a retrieval tool, updates episode memory, and stops. It cannot rewrite deterministic fields or issue trades.
- **Not a prediction or trade instruction.**

**Case packs and references:**

The runbook covers **Two momentum-crash mechanisms** — **Daniel–Moskowitz** recovery-driven reversal and **Khandani–Lo** crowded-position unwind — and the **AI evidence view** in Step 4.

Frozen/reference packs used by the demo:

| Pack | Path |
| --- | --- |
| Primary correlated-cluster case (2026-05-29) | `current_semi_unwind` → `outputs/snapshot_2026-05-29` |
| 2020 historical validation | `march_2020_reference` |
| 2024 quiet control | `quiet_control_2024` |
| Cross-case comparison | `cross_case_comparison.md` |
| Production path | `docs/production_path.md` |

</details>


**Runbook complete.** The PPT presents the story; this notebook produced the step-by-step numbers for the 2026-05-29 case and the 2026-06-30 comparison.